# 1) Forecasting basics — Exercises

**Goals:** write the metrics yourself, derive which lags a day-ahead forecast may use, make the weekly baseline aware of holidays, run a rolling-origin backtest, and blend two baselines with a weight chosen honestly.

The setup cell loads the lecture's data (`demand`, `holiday`, `train` = 2023–2024, `test` = 2025). Demand in MW.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# forecast_data.py lives in the module folder: search upwards so this works from exercise/ and exercise/solutions/
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "forecast_data.py").exists())))
from forecast_data import load_data

warnings.filterwarnings("ignore", category=FutureWarning)
df = load_data()
demand, holiday = df["demand"], df["holiday"]
train, test = demand[:"2024"], demand["2025"]

### Warm-ups

**1. The metrics**

Write `scores(forecast, actual, scale)` returning a dict with MAE, RMSE, bias, MAPE (in %) and MASE (MAE / scale), and `mase_scale(train, season=168)` — the in-sample MAE of the seasonal naive forecast on the training data.

In [2]:
def scores(forecast, actual, scale: float) -> dict:
    """MAE, RMSE, bias, MAPE (%) and MASE of a forecast."""
    ### BEGIN SOLUTION
    e = np.asarray(forecast, float) - np.asarray(actual, float)
    mae = np.abs(e).mean()
    return {"MAE": mae, "RMSE": np.sqrt((e ** 2).mean()), "bias": e.mean(),
            "MAPE_%": 100 * (np.abs(e) / np.asarray(actual, float)).mean(), "MASE": mae / scale}
    ### END SOLUTION


def mase_scale(train: pd.Series, season: int = 168) -> float:
    """MAE of the seasonal naive forecast on the training data."""
    ### BEGIN SOLUTION
    return float((train - train.shift(season)).abs().mean())
    ### END SOLUTION

In [3]:
toy = scores([110, 90, 130], [100, 100, 100], scale=10.0)
assert np.isclose(toy["MAE"], 50 / 3) and np.isclose(toy["bias"], 10.0) and np.isclose(toy["MASE"], 5 / 3)
assert np.isclose(toy["RMSE"], np.sqrt((100 + 100 + 900) / 3)) and np.isclose(toy["MAPE_%"], 50 / 3)
assert np.isclose(mase_scale(train), 129.8, atol=0.1), "the lecture's 129.8 MW"

**2. Which lags are allowed?**

A forecast for hour h (0–23) of day D is issued at `issue_hour` on day D−1, when demand is known up to the hour before `issue_hour`. Write `daily_lag(target_hour, issue_hour)` returning the smallest whole number of days k such that "the same hour k days earlier" is known — as a lag in hours (24·k). Check the lecture's rule, then: what changes for an intraday forecast issued at 06:00 on the same day D (`same_day=True`)?

In [4]:
def daily_lag(target_hour: int, issue_hour: int, same_day: bool = False) -> int:
    """Smallest lag (hours, a multiple of 24) to the same hour on an earlier day that is known at issue time."""
    ### BEGIN SOLUTION
    issue_day = 0 if same_day else -1                     # day of issue relative to the target day
    k = 1
    while True:
        known = (-k < issue_day) or (-k == issue_day and target_hour < issue_hour)
        if known:
            return 24 * k
        k += 1
    ### END SOLUTION

In [5]:
assert [daily_lag(h, 12) for h in (0, 11, 12, 23)] == [24, 24, 48, 48], "the lecture's rule"
assert daily_lag(5, 6, same_day=True) == 24 and daily_lag(18, 6, same_day=True) == 24
assert daily_lag(10, 0) == 48, "issued at midnight: nothing of the day of issue is known yet"

### Core

**3. A holiday-aware weekly baseline**

"Same hour last week" fails on holidays. Write `holiday_aware(demand, holiday, index)`: for a target hour on a **holiday**, use the same hour of the most recent Sunday before it; on a normal day whose day one week earlier **was a holiday**, use two weeks earlier; otherwise one week. Score it against the plain weekly baseline on 2025 — overall and on holidays.

In [6]:
def holiday_aware(demand: pd.Series, holiday: pd.Series, index: pd.DatetimeIndex) -> pd.Series:
    """Weekly naive forecast that treats holidays like Sundays and skips holidays as references."""
    ### BEGIN SOLUTION
    out = demand.shift(168).loc[index].copy()
    two_weeks = demand.shift(336).loc[index]
    was_holiday = holiday.shift(168).loc[index].fillna(False).astype(bool)
    out[was_holiday] = two_weeks[was_holiday]
    for t in index[holiday.loc[index].to_numpy()]:
        days_back = (t.dayofweek + 1) % 7 or 7                   # most recent Sunday strictly before t
        out[t] = demand[t - pd.Timedelta(days=days_back)]
    return out
    ### END SOLUTION

In [7]:
plain = demand.shift(168).loc[test.index]
aware = holiday_aware(demand, holiday, test.index)
h = holiday.loc[test.index]
result = pd.DataFrame({
    "plain weekly": {"MAE all": (plain - test).abs().mean(), "MAE holidays": (plain - test)[h].abs().mean()},
    "holiday-aware": {"MAE all": (aware - test).abs().mean(), "MAE holidays": (aware - test)[h].abs().mean()}}).T
assert result.loc["holiday-aware", "MAE holidays"] < result.loc["plain weekly", "MAE holidays"]
assert result.loc["holiday-aware", "MAE all"] < result.loc["plain weekly", "MAE all"]
result.round(1)

               MAE all  MAE holidays
plain weekly     142.5         189.2
holiday-aware    137.1         136.3

**4. A rolling-origin backtest**

Climatology had a bias of −29 MW in 2025. One possible reason: it was estimated once, from 2023–2024, and never updated. Test that idea. Write `rolling_climatology(demand, test_index)`: for each month of 2025, re-estimate the hour-of-week averages from **all data before that month** and forecast the month. Report the monthly bias of the fixed and the rolling version.

In [8]:
def rolling_climatology(demand: pd.Series, test_index: pd.DatetimeIndex) -> pd.Series:
    """Hour-of-week climatology re-estimated at the start of every month from all earlier data."""
    ### BEGIN SOLUTION
    parts = []
    for _, month in pd.Series(test_index, index=test_index).groupby(test_index.to_period("M")):
        past = demand[: month.index[0] - pd.Timedelta(hours=1)]
        table = past.groupby([past.index.dayofweek, past.index.hour]).mean()
        idx = month.index
        parts.append(pd.Series(table.loc[list(zip(idx.dayofweek, idx.hour))].to_numpy(), index=idx))
    return pd.concat(parts)
    ### END SOLUTION

In [9]:
fixed_table = train.groupby([train.index.dayofweek, train.index.hour]).mean()
fixed = pd.Series(fixed_table.loc[list(zip(test.index.dayofweek, test.index.hour))].to_numpy(), index=test.index)
rolling = rolling_climatology(demand, test.index)
assert rolling["2025-01"].equals(fixed["2025-01"]), "January uses exactly the training data"
monthly = pd.DataFrame({"bias fixed": (fixed - test).groupby(test.index.month).mean(),
                        "bias rolling": (rolling - test).groupby(test.index.month).mean()}).rename_axis("month")
print(f"bias over 2025: fixed {(fixed - test).mean():.1f} MW, rolling {(rolling - test).mean():.1f} MW")
monthly.round(1).T

bias over 2025: fixed -29.0 MW, rolling -16.1 MW


month            1      2      3      4      5     6     7     8      9   \
bias fixed   -248.6 -307.2 -151.5   97.3  154.8  -4.2  46.9 -37.8  151.1   
bias rolling -248.6 -297.1 -130.9  122.8  175.9  11.0  61.6 -25.3  164.9   

month           10    11    12  
bias fixed    87.2 -75.9 -78.3  
bias rolling  95.3 -70.1 -70.3  

**5. Interpretation**

What did the holiday rule fix, and what could it not fix? Did re-estimating climatology every month remove the bias — and if not, why not? What does that tell you about baselines that average over long periods?

<!-- BEGIN SOLUTION -->
The holiday rule fixes what the calendar can explain: holiday errors fall from 189.2 to 136.3 MW (−28 %), and the overall MAE from 142.5 to 137.1 MW, because a holiday now borrows a Sunday's shape and a normal day no longer copies a holiday. It cannot fix what the calendar does not know — the weather of the day, which still changes from one week to the next, and special cases such as bridge days between a holiday and a weekend.

Re-estimating climatology every month barely helps: the annual bias only falls from −29 to −16 MW. The monthly table shows why. An hour-of-week average has **no season** in it: it averages winter and summer, so it under-forecasts January and February by 250–300 MW and over-forecasts spring and autumn by up to about 175 MW, while summer — where cooling and tourism lift demand — comes out close. These errors largely cancel over the year, which is why the annual bias looked small. Adding a few more months of data cannot add the missing dimension. The lesson: a near-zero bias over a long period can hide large biases that cancel — report bias by season (or by hour, or by day type), and give a baseline the structure the data actually have.
<!-- END SOLUTION -->

### Challenge

**6. Blending two baselines — honestly**

A weighted blend w · (last week) + (1 − w) · (climatology) can beat both. Choose w from [0, 0.1, …, 1] **without looking at 2025**: estimate climatology on 2023 only, pick the w that minimises MAE on 2024, then refit climatology on 2023–2024 and evaluate that w on 2025. Compare with choosing w on 2025 itself.

In [10]:
def choose_weight(demand: pd.Series, weights) -> tuple[float, pd.Series]:
    """Weight chosen on 2024 (climatology from 2023), and the MAE on 2024 for every weight."""
    ### BEGIN SOLUTION
    fit, valid = demand[:"2023"], demand["2024"]
    table = fit.groupby([fit.index.dayofweek, fit.index.hour]).mean()
    clim = pd.Series(table.loc[list(zip(valid.index.dayofweek, valid.index.hour))].to_numpy(), index=valid.index)
    weekly = demand.shift(168).loc[valid.index]
    maes = pd.Series({w: ((w * weekly + (1 - w) * clim) - valid).abs().mean() for w in weights})
    return float(maes.idxmin()), maes
    ### END SOLUTION

In [11]:
weights = np.round(np.linspace(0, 1, 11), 1)
w_star, valid_mae = choose_weight(demand, weights)
weekly_test = demand.shift(168).loc[test.index]
test_mae = pd.Series({w: ((w * weekly_test + (1 - w) * fixed) - test).abs().mean() for w in weights})
assert 0 < w_star < 1, "a real blend, not one of the two baselines"
print(f"weight chosen on 2024: {w_star} -> 2025 MAE {test_mae[w_star]:.1f} MW "
      f"(last week alone {test_mae[1.0]:.1f}, climatology alone {test_mae[0.0]:.1f}; "
      f"best weight in hindsight {test_mae.idxmin()} -> {test_mae.min():.1f})")
pd.DataFrame({"MAE 2024 (choosing)": valid_mae, "MAE 2025 (testing)": test_mae}).round(1).T

weight chosen on 2024: 0.7 -> 2025 MAE 128.3 MW (last week alone 142.5, climatology alone 168.5; best weight in hindsight 0.6 -> 127.7)


                       0.0    0.1    0.2    0.3    0.4    0.5    0.6    0.7  \
MAE 2024 (choosing)  155.7  146.0  137.2  129.4  123.0  118.3  115.4  115.0   
MAE 2025 (testing)   168.5  157.8  148.2  140.0  133.6  129.4  127.7  128.3   

                       0.8    0.9    1.0  
MAE 2024 (choosing)  117.0  121.1  127.1  
MAE 2025 (testing)   131.1  136.0  142.5  

**7. Interpretation**

Why does a blend of two mediocre baselines beat both? Why must the weight be chosen on 2024 and not on 2025 — and how much did the honest choice cost compared with hindsight?

<!-- BEGIN SOLUTION -->
The two baselines make different kinds of errors. Last week's value carries last week's weather and noise — unbiased but noisy; climatology is smooth but blind to recent conditions and to the season. Where one is too high the other is often too low, so a weighted average cancels part of both errors: the blend reaches 128.3 MW on 2025, 10 % better than last week alone (142.5 MW) and far better than climatology (168.5 MW).

The weight is a parameter of the forecast, so it must be chosen on data the evaluation never sees. Choosing it on 2025 and then reporting its 2025 score is a leak through model selection, just like the lecture's leak through the data. Here the honest choice (0.7 from 2024) cost only 0.6 MW against the best weight in hindsight (0.6, 127.7 MW) — but the hindsight number is not a result anyone could have achieved, and with more parameters to choose the gap grows. A three-way split (fit on 2023, choose on 2024, test on 2025) is the standard pattern.
<!-- END SOLUTION -->